# Inspeção do ToolACE

Objetivo: entender as descrições das ferramentas, os pedidos e as respostas supervisionadas antes de preparar o treinamento.

A revisão abaixo foi identificada na inspeção inicial no Colab e fica fixada para reprodução. O registro dessa inspeção está em `reports/data_inspection.json`.

Fonte: https://huggingface.co/datasets/Team-ACE/ToolACE — licença declarada Apache 2.0.

Executar com CPU no Colab/Kaggle. As chamadas presentes nos dados são texto; este notebook não executa as ferramentas descritas.


In [ ]:
import json
from urllib.request import urlopen

toolace_revision = "6bda777c88d21e5a204703c1ee45597a8fa4f734"
url = (
    "https://huggingface.co/datasets/Team-ACE/ToolACE/"
    f"resolve/{toolace_revision}/data.json"
)
with urlopen(url, timeout=60) as resposta:
    toolace_data = json.load(resposta)

assert isinstance(toolace_data, list) and toolace_data
exemplo = toolace_data[0]
print("Revisão:", toolace_revision)
print("Exemplos:", len(toolace_data))
print("Campos:", list(exemplo))
print("\nINSTRUÇÕES E FERRAMENTAS:\n")
print(exemplo["system"])
for papel in ("user", "assistant"):
    mensagem = next(m for m in exemplo["conversations"] if m["from"] == papel)
    print(f"\n{papel.upper()}:\n{mensagem['value']}")


## Estrutura das conversas

Verificar tipos de mensagens, sequências e comprimentos antes de selecionar o recorte. Esta auditoria não avalia a correção das chamadas. Os resultados conferidos no Colab estão no relatório de inspeção.


In [ ]:
from collections import Counter
from statistics import median

papeis = Counter()
sequencias = Counter()
tamanhos = []
problemas = []

for indice, registro in enumerate(toolace_data):
    if not isinstance(registro, dict):
        problemas.append(indice)
        continue

    mensagens = registro.get("conversations")
    estrutura_valida = (
        isinstance(registro.get("system"), str)
        and isinstance(mensagens, list)
        and len(mensagens) > 0
    )
    if not estrutura_valida:
        problemas.append(indice)
        continue

    mensagens_validas = all(
        isinstance(mensagem, dict)
        and isinstance(mensagem.get("from"), str)
        and isinstance(mensagem.get("value"), str)
        for mensagem in mensagens
    )
    if not mensagens_validas:
        problemas.append(indice)
        continue

    sequencia = tuple(mensagem["from"] for mensagem in mensagens)
    papeis.update(sequencia)
    sequencias[sequencia] += 1
    tamanhos.append(len(mensagens))

print("Registros examinados:", len(toolace_data))
print("Registros com estrutura válida:", len(tamanhos))
print("Registros com problemas estruturais:", len(problemas))
print("\nQuantidade de mensagens por papel:")
for papel, quantidade in papeis.most_common():
    print(f"  {papel}: {quantidade}")
print("\nOito sequências de conversa mais frequentes:")
for sequencia, quantidade in sequencias.most_common(8):
    print(f"  {' → '.join(sequencia)}: {quantidade}")
if tamanhos:
    print("\nMensagens por conversa:")
    print("  Mínimo:", min(tamanhos))
    print("  Mediana:", median(tamanhos))
    print("  Máximo:", max(tamanhos))
if problemas:
    print("\nPrimeiros índices com problemas:", problemas[:10])


## Leitura das descrições de ferramentas

O primeiro colchete das instruções nem sempre inicia a lista de ferramentas: pode pertencer a um parâmetro interno. Esta etapa usa o marcador explícito da apresentação JSON. Registros com outras representações ficam fora dos candidatos atuais, sem serem classificados como inválidos. A separação das respostas por colchetes é apenas uma observação de formato.


In [ ]:
import json
from collections import Counter

marcador = "Here is a list of functions in JSON format that you can invoke:"
candidatos_json = []
diagnostico = Counter()
formatos_resposta = Counter()
erros_json = []

for indice, registro in enumerate(toolace_data):
    mensagens = registro["conversations"]
    if [m["from"] for m in mensagens] != ["user", "assistant"]:
        continue
    instrucoes = registro["system"]
    if marcador not in instrucoes:
        diagnostico["Sem o marcador esperado"] += 1
        continue
    trecho = instrucoes.split(marcador, 1)[1].lstrip()
    try:
        ferramentas, _ = json.JSONDecoder().raw_decode(trecho)
        if not isinstance(ferramentas, list) or not ferramentas:
            raise ValueError("Lista vazia ou formato diferente de lista")
        for ferramenta in ferramentas:
            if not isinstance(ferramenta, dict):
                raise ValueError("Ferramenta não é um objeto")
            if not isinstance(ferramenta.get("name"), str):
                raise ValueError("Nome ausente ou inválido")
            if not isinstance(ferramenta.get("parameters"), dict):
                raise ValueError("Parâmetros ausentes ou inválidos")
    except (ValueError, TypeError) as erro:
        diagnostico["Marcador presente, mas leitura falhou"] += 1
        erros_json.append((indice, str(erro)))
        continue
    resposta = mensagens[1]["value"].strip()
    candidatos_json.append({
        "indice_original": indice,
        "ferramentas": ferramentas,
        "pedido": mensagens[0]["value"],
        "resposta": resposta,
    })
    diagnostico["Ferramentas lidas com sucesso"] += 1
    formato = (
        "Resposta entre colchetes"
        if resposta.startswith("[") and resposta.endswith("]")
        else "Outro formato de resposta"
    )
    formatos_resposta[formato] += 1

print("Diagnóstico das 10.500 conversas curtas:")
for motivo, quantidade in diagnostico.items():
    print(f"  {motivo}: {quantidade}")
print("\nFormato das respostas nos candidatos:")
for formato, quantidade in formatos_resposta.items():
    print(f"  {formato}: {quantidade}")
if erros_json:
    print("\nPrimeiras falhas:", erros_json[:5])


## Amostra de respostas fora dos colchetes

Amostragem com semente fixa para investigar recusas por falta de ferramenta, pedidos de dados faltantes e outras possíveis representações. Os quatro casos inspecionados não definem rótulos para toda a população.


In [ ]:
import random

respostas_outros_formatos = [
    exemplo
    for exemplo in candidatos_json
    if not (
        exemplo["resposta"].startswith("[")
        and exemplo["resposta"].endswith("]")
    )
]
amostra = random.Random(42).sample(
    respostas_outros_formatos,
    min(4, len(respostas_outros_formatos)),
)
print("Total de respostas fora dos colchetes:", len(respostas_outros_formatos))
for exemplo in amostra:
    print("\n" + "=" * 60)
    print("ÍNDICE ORIGINAL:", exemplo["indice_original"])
    print("\nFERRAMENTAS DISPONÍVEIS:")
    for ferramenta in exemplo["ferramentas"]:
        print("-", ferramenta["name"])
    print("\nPEDIDO:")
    print(exemplo["pedido"][:600])
    print("\nRESPOSTA ANOTADA:")
    print(exemplo["resposta"][:1400])
    if len(exemplo["pedido"]) > 600 or len(exemplo["resposta"]) > 1400:
        print("\n[Exibição limitada; o conteúdo original foi preservado.]")
